In [ ]:
#import libraries

import os
import glob
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import cv2
from PIL import Image

import matplotlib.pyplot as plt # type: ignores

#loading bar for loops
from tqdm.auto import tqdm # type: ignore

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchinfo import summary

In [ ]:
#setting up paths for the project

print("Python home directory:", Path.home())
print("Current working directory:", Path.cwd())

print("\nContents of current directory:")
for item in Path.cwd().iterdir():
    print("  ", item)

PROJECT_ROOT = Path.cwd()

FOLDER = Path.home() / "abhista" 

DATASET_ROOT = FOLDER / "datasets" / "PRImALayoutAnalysisDataset"

IMAGE_DIR = DATASET_ROOT / "Images"

XML_DIR = DATASET_ROOT / "XML"


#display the paths for verification

print("Project root:", PROJECT_ROOT)
print("Dataset     :", DATASET_ROOT)
print("Images      :", IMAGE_DIR)
print("XML         :", XML_DIR)



print("\n--- Path Verification ---")

print("Dataset exists:", DATASET_ROOT.exists())
print("Images exists :", IMAGE_DIR.exists())
print("XML exists    :", XML_DIR.exists())



In [ ]:
#install dependencies

!%pip install -q torch torchvision torchinfo lxml opencv-python-headless pillow matplotlib tqdm scikit-learn

In [ ]:
# Set the random seed for reproducibility

SEED = 142

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Set the device to GPU if available, otherwise use CPU

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#inspect the dataset

print("Dataset contents:")

# Check if dataset exists first
if DATASET_ROOT.exists():
    for item in DATASET_ROOT.iterdir():
        path = DATASET_ROOT / item

        if path.is_dir():
            print("[DIR ]", item)
        else:
            print("[FILE]", item)
else:
    print(f"Error: Dataset directory does not exist at {DATASET_ROOT}")
    print("Please verify the dataset path or download/mount the dataset.")

In [ ]:
#verifying the paths and checking folder sizes

print("Dataset exists:", DATASET_ROOT.exists())
print("Images exists :", IMAGE_DIR.exists())
print("XML exists    :", XML_DIR.exists())
print("Number of image files:", len(list(IMAGE_DIR.rglob("*"))))

print("Number of XML files:", len(list(XML_DIR.rglob("*.xml"))))

In [ ]:
print("\nImages:")
print(glob.glob(os.path.join(IMAGE_DIR, "*"))[2:4])

print("\nXML files:")
print(glob.glob(os.path.join(XML_DIR, "*"))[2:4])

In [ ]:
#defining the classes for segmentation

CLASS_NAMES = [
    "background",
    "text",
    "image",
    "line_drawing",
    "graphic",
    "table",
    "chart",
    "separator",
    "maths",
    "noise",
    "frame"
]

CLASS_TO_ID = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

ID_TO_CLASS = {
    idx: name
    for name, idx in CLASS_TO_ID.items()
}

NUM_CLASSES = len(CLASS_NAMES)

print("Number of classes:", NUM_CLASSES)
print(CLASS_TO_ID)
# print(ID_TO_CLASS)

In [ ]:
# Helper function to extract local name from XML tags

def get_local_name(tag):
    """
    Removes XML namespace from a tag.

    Example:
        {http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15}TextRegion

    becomes:
        TextRegion
    """
    return tag.split("}")[-1]

In [ ]:
#defining the region types and mapping for segmentation

REGION_TYPES = {
    "TextRegion": "text",
    "ImageRegion": "image",
    "LineDrawingRegion": "line_drawing",
    "GraphicRegion": "graphic",
    "TableRegion": "table",
    "ChartRegion": "chart",
    "SeparatorRegion": "separator",
    "MathsRegion": "maths",
    "NoiseRegion": "noise",
    "FrameRegion": "frame",
}

In [ ]:
#parse polygon co-ordinates

def parse_points(points_string):
    """
    Convert PAGE XML coordinate points
    originally in string
    into an Nx2 numpy array.

    Example:
        "10,20 100,20 100,200 10,200"

    returns:
        [[ 10,  20],
         [100,  20],
         [100, 200],
         [ 10, 200]]
    """

    points = []

    if points_string is None:
        return np.empty((0, 2), dtype=np.int32)

    for point in points_string.strip().split():

        if "," not in point:
            continue

        x, y = point.split(",")[:2]

        try:
            x = float(x)
            y = float(y)

            points.append([int(round(x)), int(round(y))])

        except ValueError:
            continue

    return np.array(points, dtype=np.int32)

In [ ]:
#extract polygon co-ordinates from PAGE XML region

def extract_region_polygon(region_element):
    """
    Extract polygon coordinates from a PAGE XML region.
    """

    for child in region_element.iter():

        if get_local_name(child.tag) == "Coords":

            points = []

            for point in child:

                if get_local_name(point.tag) == "Point":

                    x = point.attrib.get("x")
                    y = point.attrib.get("y")

                    if x is not None and y is not None:
                        points.append([
                            int(round(float(x))),
                            int(round(float(y)))
                        ])

            if len(points) >= 3:
                return np.array(points, dtype=np.int32)

            # Some PAGE XML files may use a points attribute
            points_string = child.attrib.get("points")

            if points_string:
                points = parse_points(points_string)

                if len(points) >= 3:
                    return points

    return np.empty((0, 2), dtype=np.int32)

In [ ]:
#convert PAGE XML annotations into a semantic segmentation mask

def xml_to_mask(xml_path, image_width, image_height):
    """
    Convert PAGE XML annotations into a semantic segmentation mask.

    Output:
        H x W uint8 array
        where each pixel contains a class ID.
    """

    mask = np.zeros(
        (image_height, image_width),
        dtype=np.uint8
    )

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for element in root.iter():

        tag = get_local_name(element.tag)

        if tag not in REGION_TYPES:
            continue
            

        class_name = REGION_TYPES[tag]

        class_id = CLASS_TO_ID[class_name]

        polygon = extract_region_polygon(element)

        if polygon.shape[0] < 3:
            continue

        # Clamp coordinates to image boundaries
        polygon[:, 0] = np.clip(
            polygon[:, 0],
            0,
            image_width - 1
        )

        polygon[:, 1] = np.clip(
            polygon[:, 1],
            0,
            image_height - 1
        )

        cv2.fillPoly(
            mask,
            [polygon],
            class_id
        )

    return mask

In [ ]:
#create pairs of image and corresponding XML files based on their filenames

def create_image_xml_pairs(image_dir, xml_dir):

    image_extensions = [
        "*.jpg",
        "*.jpeg",
        "*.png",
        "*.tif",
        "*.tiff",
        "*.bmp"
    ]

    image_files = []

    for ext in image_extensions:
        image_files.extend(
            glob.glob(os.path.join(image_dir, ext))
        )

    xml_files = glob.glob(
        os.path.join(xml_dir, "*.xml")
    )

    xml_lookup = {
        Path(x).stem.lower().removeprefix("pc-"): x
        for x in xml_files
    }

    pairs = []

    for image_path in image_files:

        stem = Path(image_path).stem.lower()

        if stem in xml_lookup:

            pairs.append(
                (
                    image_path,
                    xml_lookup[stem]
                )
            )

    return sorted(pairs)

In [ ]:
#sample the first 10 image/XML pairs for verification

pairs = create_image_xml_pairs(
    IMAGE_DIR,
    XML_DIR
)
print(IMAGE_DIR, XML_DIR)
print("Number of image/XML pairs:", len(pairs))

for image_path, xml_path in pairs[:10]:
    print(
        Path(image_path).name,
        "<->",
        Path(xml_path).name
    )

In [ ]:
#images without corresponding XML files

def get_images_without_xml(image_dir, xml_dir):

    image_extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".tif",
        ".tiff",
        ".bmp"
    }

    image_files = [
        path
        for path in Path(image_dir).iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    ]

    xml_stems = {
        path.stem.lower().removeprefix("pc-")
        for path in Path(xml_dir).glob("*.xml")
    }

    ignored_images = {
        str(path)
        for path in image_files
        if path.stem.lower().removeprefix("pc-") not in xml_stems
    }

    return sorted(ignored_images)


ignored_images = get_images_without_xml(
    IMAGE_DIR,
    XML_DIR
)

print("Number of images without XML files:", len(ignored_images))

for image_path in ignored_images:
    print(image_path)

In [ ]:
#verify whether at least one image/XML pair was found

if len(pairs) == 0:
    raise RuntimeError(
        "No matching image/XML pairs were found. "
        "Check IMAGE_DIR, XML_DIR and filename matching."
    )

print("Dataset successfully paired.")

In [ ]:
#storing generated masks separately

MASK_DIR = os.path.join(
    DATASET_ROOT,
    "generated_masks"
)

os.makedirs(MASK_DIR, exist_ok=True)

print("Mask directory:", MASK_DIR)

In [ ]:
#generating masks

def get_image_size(image_path):

    with Image.open(image_path) as img:
        return img.size


for image_path, xml_path in tqdm(
    pairs,
    desc="Generating masks"
):

    width, height = get_image_size(image_path)

    mask = xml_to_mask(
        xml_path,
        width,
        height
    )

    mask_path = os.path.join(
        MASK_DIR,
        Path(image_path).stem + ".png"
    )

    Image.fromarray(mask).save(mask_path)

In [ ]:
#verifying generated masks

class_pixel_counts = np.zeros(
    NUM_CLASSES,
    dtype=np.int64
)

for mask_path in glob.glob(
    os.path.join(MASK_DIR, "*.png")
):

    mask = np.array(
        Image.open(mask_path)
    )

    values, counts = np.unique(
        mask,
        return_counts=True
    )

    for value, count in zip(values, counts):

        if value < NUM_CLASSES:
            class_pixel_counts[value] += count


for class_id, count in enumerate(class_pixel_counts):

    print(
        f"{class_id:2d} "
        f"{ID_TO_CLASS[class_id]:15s} "
        f"{count:,}"
    )

In [ ]:
#function to visualize generated masks

def visualize_mask(mask):

    plt.figure(figsize=(12, 8))

    plt.imshow(mask)

    plt.title("Segmentation Mask")

    plt.axis("off")

    plt.show()

In [ ]:
#example of visualizing a generated mask

sample_mask_path = glob.glob(
    os.path.join(MASK_DIR, "*.png")
)[4]

sample_mask = np.array(
    Image.open(sample_mask_path)
)

visualize_mask(sample_mask)

In [ ]:
#function to visualize image and mask together

def visualize_sample(image_path, mask_path):

    image = Image.open(image_path).convert("RGB")

    mask = np.array(
        Image.open(mask_path)
    )

    plt.figure(figsize=(10, 6))

    plt.subplot(1, 2, 1)

    plt.imshow(image)

    plt.title("Original Image")

    plt.axis("off")

    plt.subplot(1, 2, 2)

    plt.imshow(mask)

    plt.title("Ground Truth Mask")

    plt.axis("off")

    plt.show()

In [ ]:
#example of visualizing a sample image and its corresponding mask

sample_image_path, sample_xml_path = pairs[5]

sample_mask_path = os.path.join(
    MASK_DIR,
    Path(sample_image_path).stem + ".png"
)

visualize_sample(
    sample_image_path,
    sample_mask_path
)

In [ ]:
#final dataset

dataset_items = []

for image_path, xml_path in pairs:

    mask_path = os.path.join(
        MASK_DIR,
        Path(image_path).stem + ".png"
    )

    if os.path.exists(mask_path):

        dataset_items.append({
            "image": image_path,
            "xml": xml_path,
            "mask": mask_path
        })

print("Total usable samples:", len(dataset_items))

In [ ]:
#train,test and validation = 70%, 15%, 15%

train_items, temp_items = train_test_split(
    dataset_items,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

val_items, test_items = train_test_split(
    temp_items,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Training:", len(train_items))
print("Validation:", len(val_items))
print("Testing:", len(test_items))

In [ ]:
#resizing images

IMAGE_SIZE = 512

BATCH_SIZE = 4

NUM_WORKERS = 2

LEARNING_RATE = 1e-4

NUM_EPOCHS = 30

WEIGHT_DECAY = 1e-5

CHECKPOINT_PATH = FOLDER / "datasets" / "best_unet_prima.pth"

# print("Checkpoint path:", CHECKPOINT_PATH)

#!ls -lh {CHECKPOINT_PATH}

In [ ]:
#building the dataset class

class PRImADataset(Dataset):

    def __init__(
        self,
        items,
        image_size=512,
        augment=False
    ):

        self.items = items

        self.image_size = image_size

        self.augment = augment

    def __len__(self):

        return len(self.items)

    def __getitem__(self, index):

        item = self.items[index]

        image = Image.open(
            item["image"]
        ).convert("RGB")

        mask = Image.open(
            item["mask"]
        )

        image = np.array(image)

        mask = np.array(mask)

        # Resize image
        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

        # Resize segmentation mask
        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        # Data augmentation
        if self.augment:

            if random.random() < 0.5:

                image = np.fliplr(
                    image
                ).copy()

                mask = np.fliplr(
                    mask
                ).copy()

            if random.random() < 0.5:

                image = np.flipud(
                    image
                ).copy()

                mask = np.flipud(
                    mask
                ).copy()

        # Convert image to float
        image = image.astype(
            np.float32
        ) / 255.0

        # HWC -> CHW
        image = np.transpose(
            image,
            (2, 0, 1)
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        )

        mask = torch.tensor(
            mask,
            dtype=torch.long
        )

        return image, mask

In [ ]:
#create dataset instances for training, validation, and testing

train_dataset = PRImADataset(
    train_items,
    image_size=IMAGE_SIZE,
    augment=True
)

val_dataset = PRImADataset(
    val_items,
    image_size=IMAGE_SIZE,
    augment=False
)

test_dataset = PRImADataset(
    test_items,
    image_size=IMAGE_SIZE,
    augment=False
)

In [ ]:
#defining dataloader class

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
#verifying dataloader

images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

print("Image min:", images.min().item())
print("Image max:", images.max().item())

print("Mask classes:", torch.unique(masks))

In [ ]:
#importing the UNet model from the uNet_Model.py file

import importlib
import uNet_Model

uNet_Model = importlib.reload(uNet_Model)

from uNet_Model import DoubleConv, DownBlock, UpBlock, UNet

In [ ]:
#initializing the model

model = UNet(
    in_channels=3,
    num_classes=NUM_CLASSES
).to(DEVICE)

print(model)

In [ ]:
#model summary

summary(
    model,
    input_size=(
        1,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    ),
    col_names=[
        "input_size",
        "output_size",
        "num_params",
        "mult_adds"
    ],
    depth=4
)

In [ ]:
#calculate Dice loss

class DiceLoss(nn.Module):

    def __init__(
        self,
        smooth=1.0
    ):

        super().__init__()

        self.smooth = smooth

    def forward(
        self,
        logits,
        targets
    ):

        num_classes = logits.shape[1]

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        targets_one_hot = F.one_hot(
            targets,
            num_classes=num_classes
        )

        targets_one_hot = targets_one_hot.permute(
            0,
            3,
            1,
            2
        ).float()

        intersection = (
            probabilities *
            targets_one_hot
        ).sum(
            dim=(0, 2, 3)
        )

        denominator = (
            probabilities.sum(
                dim=(0, 2, 3)
            )
            +
            targets_one_hot.sum(
                dim=(0, 2, 3)
            )
        )

        dice = (
            2.0 * intersection +
            self.smooth
        ) / (
            denominator +
            self.smooth
        )

        return 1.0 - dice.mean()

In [ ]:
#calculate combined loss
class CombinedLoss(nn.Module):

    def __init__(
        self,
        ce_weight=0.5,
        dice_weight=0.5
    ):

        super().__init__()

        self.ce = nn.CrossEntropyLoss()

        self.dice = DiceLoss()

        self.ce_weight = ce_weight

        self.dice_weight = dice_weight

    def forward(
        self,
        logits,
        targets
    ):

        ce_loss = self.ce(
            logits,
            targets
        )

        dice_loss = self.dice(
            logits,
            targets
        )

        total_loss = (
            self.ce_weight * ce_loss
            +
            self.dice_weight * dice_loss
        )

        return total_loss

In [ ]:
#optimizer

criterion = CombinedLoss(
    ce_weight=0.5,
    dice_weight=0.5
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
#learning rate scheduler

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

In [ ]:
"""
Calculates:
Pixel Accuracy
Mean IoU
Mean Dice
Per-class IoU
Per-class Dice
"""

#metric function

def calculate_metrics(
    predictions,
    targets,
    num_classes
):

    predictions = predictions.view(-1)

    targets = targets.view(-1)

    ious = []
    dices = []

    for class_id in range(num_classes):

        pred_class = (
            predictions == class_id
        )

        target_class = (
            targets == class_id
        )

        intersection = (
            pred_class &
            target_class
        ).sum().item()

        union = (
            pred_class |
            target_class
        ).sum().item()

        pred_area = pred_class.sum().item()

        target_area = target_class.sum().item()

        if union == 0:

            iou = float("nan")

        else:

            iou = (
                intersection /
                union
            )

        denominator = (
            pred_area +
            target_area
        )

        if denominator == 0:

            dice = float("nan")

        else:

            dice = (
                2.0 * intersection /
                denominator
            )

        ious.append(iou)

        dices.append(dice)

    valid_ious = [
        x for x in ious
        if not np.isnan(x)
    ]

    valid_dices = [
        x for x in dices
        if not np.isnan(x)
    ]

    accuracy = (
        predictions == targets
    ).float().mean().item()

    mean_iou = np.mean(
        valid_ious
    )

    mean_dice = np.mean(
        valid_dices
    )

    return {
        "accuracy": accuracy,
        "mean_iou": mean_iou,
        "mean_dice": mean_dice,
        "iou_per_class": ious,
        "dice_per_class": dices
    }


In [ ]:
#training function

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    running_loss = 0.0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, masks in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images)

        loss = criterion(
            logits,
            masks
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss /
        len(loader.dataset)
    )

    return epoch_loss

In [ ]:
#validation function

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device,
    num_classes
):

    model.eval()

    running_loss = 0.0

    all_predictions = []

    all_targets = []

    for images, masks in tqdm(
        loader,
        desc="Validation",
        leave=False
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        loss = criterion(
            logits,
            masks
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            masks.cpu()
        )

    predictions = torch.cat(
        all_predictions
    )

    targets = torch.cat(
        all_targets
    )

    metrics = calculate_metrics(
        predictions,
        targets,
        num_classes
    )

    metrics["loss"] = (
        running_loss /
        len(loader.dataset)
    )

    return metrics

In [ ]:
#training loop

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_iou": [],
    "val_dice": []
}

best_dice = -1.0

NUM_EPOCHS = 3

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):

    print(
        f"\nEpoch "
        f"{epoch + 1}/{NUM_EPOCHS}"
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )

    val_metrics = validate(
        model,
        val_loader,
        criterion,
        DEVICE,
        NUM_CLASSES
    )

    val_loss = val_metrics["loss"]

    val_accuracy = (
        val_metrics["accuracy"]
    )

    val_iou = (
        val_metrics["mean_iou"]
    )

    val_dice = (
        val_metrics["mean_dice"]
    )

    scheduler.step(
        val_dice
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    history["val_iou"].append(
        val_iou
    )

    history["val_dice"].append(
        val_dice
    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Val Acc    : {val_accuracy:.4f}"
    )

    print(
        f"Val IoU    : {val_iou:.4f}"
    )

    print(
        f"Val Dice   : {val_dice:.4f}"
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"LR         : {current_lr:.7f}"
    )

    # Save best model
    
    if val_dice > best_dice:

        best_dice = val_dice

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_dice,
                "val_iou": val_iou,
                "class_names": CLASS_NAMES,
                "image_size": IMAGE_SIZE
            },
            CHECKPOINT_PATH
        )

        print(
            f"Best model saved: "
            f"{CHECKPOINT_PATH}"
        )

In [ ]:
#loading the model with feature extraction support

import importlib
import uNet_Model

uNet_Model = importlib.reload(uNet_Model)

from uNet_Model import (
    DoubleConv,
    DownBlock,
    UpBlock,
    UNet
)

print("U-Net model module reloaded successfully.")

In [ ]:
#save the final model

FINAL_MODEL_PATH = os.path.join(
    DATASET_ROOT,
    "prima_unet_final.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "class_to_id": CLASS_TO_ID,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES
    },
    FINAL_MODEL_PATH
)

print(
    "Final model saved to:",
    FINAL_MODEL_PATH
)

In [ ]:
#load the trained U-Net model for inference

feature_model = UNet(
    in_channels=3,
    num_classes=NUM_CLASSES
).to(DEVICE)

checkpoint = torch.load(
    FINAL_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

feature_model.load_state_dict(
    checkpoint["model_state_dict"]
)

feature_model.eval()

print("Trained U-Net loaded successfully.")
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

In [ ]:
#verify the model with a sample from the test set

images, masks = next(iter(test_loader))

images = images.to(DEVICE)

with torch.no_grad():

    logits, feature_map = feature_model(
        images,
        return_features=True
    )

print("Logits shape:")
print(logits.shape)

print("Feature map shape:")
print(feature_map.shape)

In [ ]:
# ============================================================
# MASK-GUIDED DEEP FEATURE EXTRACTION
# ============================================================

@torch.no_grad()
def extract_class_features(
    model,
    image_tensor,
    segmentation_mask,
    num_classes,
    device
):
    """
    Extract one deep feature vector for each class present
    in a document image.

    Parameters
    ----------
    model:
        Trained U-Net.

    image_tensor:
        Tensor of shape [B, 3, H, W].

    segmentation_mask:
        Tensor of shape [B, H, W].
        Each pixel contains a class ID.

    num_classes:
        Number of segmentation classes.

    device:
        CPU or CUDA.

    Returns
    -------
    class_features:
        Tensor [B, num_classes, 1024].

        If a class is absent from an image, its vector is NaN.

    class_present:
        Tensor [B, num_classes].
        True if the class occurs in the image.
    """

    model.eval()

    image_tensor = image_tensor.to(device)
    segmentation_mask = segmentation_mask.to(device)

    #obtain logits and feature map from the model

    logits, feature_map = model(
        image_tensor,
        return_features=True
    )

    # feature_map:
    # [B, 1024, Hf, Wf]

    B, C, Hf, Wf = feature_map.shape

    #resize segmentation mask to feature-map resolution

    resized_mask = F.interpolate(
        segmentation_mask.unsqueeze(1).float(),
        size=(Hf, Wf),
        mode="nearest"
    ).squeeze(1).long()

    #storage

    class_features = torch.full(
        (B, num_classes, C),
        float("nan"),
        device=device
    )

    class_present = torch.zeros(
        (B, num_classes),
        dtype=torch.bool,
        device=device
    )

    
    #extract features for each class present in the image
    for class_id in range(num_classes):

        # Binary mask for this class
        class_mask = (
            resized_mask == class_id
        )

        # Number of pixels belonging to class
        pixel_count = class_mask.sum(
            dim=(1, 2)
        )

        present = pixel_count > 0

        class_present[:, class_id] = present

        #mask guided global average pooling of the feature map
        mask_float = class_mask.float()

        masked_features = (
            feature_map *
            mask_float.unsqueeze(1)
        )

        feature_sum = masked_features.sum(
            dim=(2, 3)
        )

        denominator = pixel_count.clamp(
            min=1
        ).unsqueeze(1).float()

        pooled_features = (
            feature_sum /
            denominator
        )

        class_features[:, class_id, :] = torch.where(
            present.unsqueeze(1),
            pooled_features,
            torch.full_like(
                pooled_features,
                float("nan")
            )
        )

    return (
        class_features,
        class_present
    )

In [ ]:
#plot training curves

plt.figure(figsize=(10, 6))

plt.plot(
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training and Validation Loss")

plt.legend()

plt.grid()

plt.show()

In [ ]:
#plot validation dice

plt.figure(figsize=(10, 6))

plt.plot(
    history["val_dice"],
    label="Validation Dice"
)

plt.xlabel("Epoch")

plt.ylabel("Dice")

plt.title("Validation Dice Score")

plt.legend()

plt.grid()

plt.show()

In [ ]:
#plot validation IOU

plt.figure(figsize=(10, 6))

plt.plot(
    history["val_iou"],
    label="Validation IoU"
)

plt.xlabel("Epoch")

plt.ylabel("IoU")

plt.title("Validation Mean IoU")

plt.legend()

plt.grid()

plt.show()

In [ ]:
#save training history to CSV

import pandas as pd

history_df = pd.DataFrame(
    history
)

HISTORY_PATH = os.path.join(
    DATASET_ROOT,
    "unet_training_history.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print(
    "Training history saved:",
    HISTORY_PATH
)

#load the best U-Net
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(
    "Loaded best model from epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation Dice:",
    checkpoint["val_dice"]
)


In [ ]:
#evaluation on test set

test_metrics = validate(
    model,
    test_loader,
    criterion,
    DEVICE,
    NUM_CLASSES
)

print("\n========== TEST RESULTS ==========")

print(
    f"Test Loss      : "
    f"{test_metrics['loss']:.4f}"
)

print(
    f"Pixel Accuracy : "
    f"{test_metrics['accuracy']:.4f}"
)

print(
    f"Mean IoU       : "
    f"{test_metrics['mean_iou']:.4f}"
)

print(
    f"Mean Dice      : "
    f"{test_metrics['mean_dice']:.4f}"
)

In [ ]:
#per class results

print("\n=========== PER-CLASS RESULTS =============")

for class_id, class_name in enumerate(
    CLASS_NAMES
):

    iou = test_metrics[
        "iou_per_class"
    ][class_id]

    dice = test_metrics[
        "dice_per_class"
    ][class_id]

    print(
        f"{class_name:15s} "
        f"IoU={iou:.3f} \t"
        f"Dice={dice:.3f}"
    )

In [ ]:
#prediction function

@torch.no_grad()
def predict_image(
    model,
    image_path,
    image_size,
    device
):

    model.eval()

    original = Image.open(
        image_path
    ).convert("RGB")

    original_np = np.array(
        original
    )

    resized = cv2.resize(
        original_np,
        (image_size, image_size),
        interpolation=cv2.INTER_AREA
    )

    image = (
        resized.astype(
            np.float32
        ) / 255.0
    )

    image = np.transpose(
        image,
        (2, 0, 1)
    )

    tensor = torch.tensor(
        image,
        dtype=torch.float32
    ).unsqueeze(0)

    tensor = tensor.to(device)

    logits = model(tensor)

    prediction = torch.argmax(
        logits,
        dim=1
    )

    prediction = prediction[
        0
    ].cpu().numpy()

    return original_np, prediction

In [ ]:
#inference

test_image_path = test_items[0]["image"]

original, prediction = predict_image(
    model,
    test_image_path,
    IMAGE_SIZE,
    DEVICE
)

print("Prediction shape:", prediction.shape)
print(
    "Predicted classes:",
    np.unique(prediction)
)

In [ ]:
#prediction visualization

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)

plt.imshow(original)

plt.title("Original")

plt.axis("off")


plt.subplot(1, 3, 2)

plt.imshow(prediction)

plt.title("Predicted Segmentation")

plt.axis("off")


ground_truth = np.array(
    Image.open(
        test_items[0]["mask"]
    )
)

ground_truth = cv2.resize(
    ground_truth,
    (IMAGE_SIZE, IMAGE_SIZE),
    interpolation=cv2.INTER_NEAREST
)

plt.subplot(1, 3, 3)

plt.imshow(ground_truth)

plt.title("Ground Truth")

plt.axis("off")

plt.tight_layout()

plt.show()

print("Original image file name:", Path(test_image_path).name)

In [ ]:
#comparison and overlay of prediction with original image

def create_segmentation_overlay(
    image,
    mask,
    alpha=0.45
):

    image = image.astype(
        np.uint8
    )

    overlay = np.zeros_like(
        image
    )

    # Generate a deterministic color for each class
    rng = np.random.default_rng(42)

    colors = rng.integers(
        0,
        256,
        size=(NUM_CLASSES, 3),
        dtype=np.uint8
    )

    colors[0] = [0, 0, 0]

    for class_id in range(
        1,
        NUM_CLASSES
    ):

        overlay[
            mask == class_id
        ] = colors[class_id]

    result = cv2.addWeighted(
        image,
        1 - alpha,
        overlay,
        alpha,
        0
    )

    return result

In [ ]:
#overlay display

overlay = create_segmentation_overlay(
    cv2.resize(
        original,
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    prediction
)

plt.figure(figsize=(12, 8))

plt.imshow(overlay)

plt.title(
    "U-Net Segmentation Overlay"
)

plt.axis("off")

plt.show()

In [ ]:
#generate proper class legend

def show_class_legend():

    fig, ax = plt.subplots(
        figsize=(3,3)
    )

    rng = np.random.default_rng(42)

    colors = rng.integers(
        0,
        256,
        size=(NUM_CLASSES, 3),
        dtype=np.uint8
    )

    colors[0] = [0, 0, 0]

    for class_id, class_name in enumerate(
        CLASS_NAMES
    ):

        ax.scatter(
            [],
            [],
            color=colors[class_id]/ 255.0,
            label=f"{class_id}: {class_name}"
        )

    ax.legend(
        loc="center",
        frameon=True
    )

    ax.axis("off")

    plt.show()

In [ ]:
show_class_legend()

In [ ]:
#save predictions

PREDICTION_DIR = os.path.join(
    DATASET_ROOT,
    "predictions"
)

os.makedirs(
    PREDICTION_DIR,
    exist_ok=True
)

In [ ]:
#save masks for the entire test set

model.eval()

for item in tqdm(
    test_items,
    desc="Generating test predictions"
):

    image = Image.open(
        item["image"]
    ).convert("RGB")

    image_np = np.array(
        image
    )

    resized = cv2.resize(
        image_np,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_AREA
    )

    tensor = (
        resized.astype(
            np.float32
        ) / 255.0
    )

    tensor = np.transpose(
        tensor,
        (2, 0, 1)
    )

    tensor = torch.tensor(
        tensor,
        dtype=torch.float32
    ).unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        logits = model(tensor)

        prediction = torch.argmax(
            logits,
            dim=1
        )[0].cpu().numpy()

    output_path = os.path.join(
        PREDICTION_DIR,
        Path(item["image"]).stem + ".png"
    )

    Image.fromarray(
        prediction.astype(np.uint8)
    ).save(
        output_path
    )

print(
    "Predictions saved to:",
    PREDICTION_DIR
)

In [ ]:
#visualize multiple test predictions

def visualize_predictions(
    model,
    items,
    num_samples=5
):

    num_samples = min(
        num_samples,
        len(items)
    )

    selected_items = random.sample(
        items,
        num_samples
    )

    for item in selected_items:

        original, prediction = predict_image(
            model,
            item["image"],
            IMAGE_SIZE,
            DEVICE
        )

        ground_truth = np.array(
            Image.open(
                item["mask"]
            )
        )

        ground_truth = cv2.resize(
            ground_truth,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        resized_original = cv2.resize(
            original,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        overlay = create_segmentation_overlay(
            resized_original,
            prediction
        )

        plt.figure(figsize=(18, 6))

        plt.subplot(1, 3, 1)

        plt.imshow(resized_original)

        plt.title("Original")

        plt.axis("off")

        plt.subplot(1, 3, 2)

        plt.imshow(ground_truth)

        plt.title("Ground Truth")

        plt.axis("off")

        plt.subplot(1, 3, 3)

        plt.imshow(overlay)

        plt.title("U-Net Prediction")

        plt.axis("off")

        plt.tight_layout()

        plt.show()

In [ ]:
#visualize multiple test predictions

show_class_legend() # Display the class legend
visualize_predictions(
    model,
    test_items,
    num_samples=6
)

In [ ]:
#evaluation table

results = []

for class_id, class_name in enumerate(
    CLASS_NAMES
):

    results.append({
        "Class": class_name,
        "IoU": test_metrics[
            "iou_per_class"
        ][class_id],
        "Dice": test_metrics[
            "dice_per_class"
        ][class_id]
    })

results_df = pd.DataFrame(
    results
)
#print(results)
results_df

In [ ]:
#load the model later

loaded_model = UNet(
    in_channels=3,
    num_classes=NUM_CLASSES
).to(DEVICE)

checkpoint = torch.load(
    FINAL_MODEL_PATH,
    map_location=DEVICE
)

loaded_model.load_state_dict(
    checkpoint["model_state_dict"]
)

loaded_model.eval()

print("U-Net loaded successfully.")

In [ ]:
#save training history

history_df = pd.DataFrame(
    history
)

HISTORY_PATH = os.path.join(
    DATASET_ROOT,
    "unet_training_history.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print(
    "Training history saved:",
    HISTORY_PATH
)

In [ ]:
#save evaluation results

RESULTS_PATH = os.path.join(
    DATASET_ROOT,
    "unet_test_results.csv"
)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "Results saved:",
    RESULTS_PATH
)

In [ ]:
import importlib
import uNet_Model

uNet_Model = importlib.reload(uNet_Model)

UNet = uNet_Model.UNet
import inspect

print(inspect.signature(UNet.forward))

In [ ]:
#Testing the feature extraction function for a batch of images

images, masks = next(iter(test_loader))

class_features, class_present = extract_class_features(
    feature_model,
    images,
    masks,
    NUM_CLASSES,
    DEVICE
)

print("Images shape:")
print(images.shape)

print("\nMasks shape:")
print(masks.shape)

print("\nClass feature shape:")
print(class_features.shape)

print("\nClass presence shape:")
print(class_present.shape)

In [ ]:
#inspecting the feature vector for the "text" class

text_id = CLASS_TO_ID["text"]

text_vector = class_features[0, text_id]

print("Text feature vector shape:")
print(text_vector.shape)

print("\nFirst 20 values:")
print(text_vector[:20])

In [ ]:
#display class feature vectors for the first image in the batch

for class_id, class_name in enumerate(CLASS_NAMES):

    if class_present[0, class_id]:

        vector = class_features[
            0,
            class_id
        ]

        print(
            f"{class_name:15s} "
            f"→ {vector.shape}"
        )

    else:

        print(
            f"{class_name:15s} "
            f"→ ABSENT"
        )

In [ ]:
#extracting vectors from the entire test set

all_class_features = []

all_class_presence = []

all_image_names = []

feature_model.eval()

with torch.no_grad():

    for images, masks in tqdm(
        test_loader,
        desc="Extracting deep class features"
    ):

        class_features, class_present = (
            extract_class_features(
                feature_model,
                images,
                masks,
                NUM_CLASSES,
                DEVICE
            )
        )

        all_class_features.append(
            class_features.cpu()
        )

        all_class_presence.append(
            class_present.cpu()
        )

# Combine batches

all_class_features = torch.cat(
    all_class_features,
    dim=0
)

all_class_presence = torch.cat(
    all_class_presence,
    dim=0
)

print(
    "All class features:",
    all_class_features.shape #displays [no.of images , no.of classes, feature vector length]
)

print(
    "All class presence:",
    all_class_presence.shape #displays [no.of images , no.of classes]
)

In [ ]:
#calculate class prototypes

class_prototypes = {}

for class_id, class_name in enumerate(CLASS_NAMES):

    # Select vectors where this class exists
    vectors = all_class_features[
        all_class_presence[:, class_id],
        class_id,
        :
    ]

    if vectors.shape[0] == 0:

        print(
            f"{class_name:15s}: no samples found"
        )

        continue

    # Mean prototype
    prototype = torch.nanmean(
        vectors,
        dim=0
    )

    class_prototypes[class_name] = prototype

    print(
        f"{class_name:15s}: "
        f"{vectors.shape[0]:4d} samples → "
        f"{prototype.shape[0]}-D prototype"
    )

In [ ]:
#saving the feature vectors representations and class prototypes to disk


FEATURE_DIR = Path(DATASET_ROOT) / "deep_features"

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save all document/class features

torch.save(
    {
        "features": all_class_features,
        "class_presence": all_class_presence,
        "class_names": CLASS_NAMES,
        "feature_dimension": all_class_features.shape[-1],
        "feature_source": "U-Net bottleneck",
        "image_size": IMAGE_SIZE
    },
    FEATURE_DIR / "test_class_features.pt"
)

# Save class prototypes

torch.save(
    {
        "prototypes": class_prototypes,
        "class_names": CLASS_NAMES,
        "feature_dimension": 1024,
        "feature_source": "U-Net bottleneck"
    },
    FEATURE_DIR / "class_prototypes.pt"
)

print(
    "Feature representations saved to:",
    FEATURE_DIR
)

In [ ]:
#convert prototypes to a CSV

prototype_rows = []

for class_name, vector in class_prototypes.items():

    vector_np = vector.numpy()

    row = {
        "class": class_name
    }

    for i, value in enumerate(vector_np):

        row[f"feature_{i+1}"] = value

    prototype_rows.append(row)


prototype_df = pd.DataFrame(
    prototype_rows
)

PROTOTYPE_CSV = (
    FEATURE_DIR /
    "class_prototypes_1024D.csv"
)

prototype_df.to_csv(
    PROTOTYPE_CSV,
    index=False
)

print(
    "Prototype CSV saved:",
    PROTOTYPE_CSV
)

print(
    "Shape:",
    prototype_df.shape
)

In [ ]:
#class to class cosine similarity matrix

from torch.nn.functional import cosine_similarity

prototype_names = list(
    class_prototypes.keys()
)

print("Prototype names:", prototype_names)
print("Absent classes:", set(CLASS_NAMES) - set(prototype_names))

prototype_matrix = torch.stack([
    class_prototypes[name]
    for name in prototype_names
])

# Normalize
prototype_matrix = F.normalize(
    prototype_matrix,
    p=2,
    dim=1
)

similarity_matrix = (
    prototype_matrix @
    prototype_matrix.T
)

similarity_df = pd.DataFrame(
    similarity_matrix.numpy(),
    index=prototype_names,
    columns=prototype_names
)

similarity_df